In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

In [2]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
max_length = 128
output_dir = "./my_qwen_lora"

In [3]:
raw_data = load_dataset("json", data_files="data/train.jsonl")
raw_data

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 14
    })
})

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def format_example(sample):
    instruction = sample.get("instruction", "")
    input_text = sample.get("input", "")
    output_text = sample.get("output", "")

    if input_text.strip():
        text = f"Instruction: {instruction}\nInput: {input_text}\nResponse: {output_text}"
    else:
        text = f"Instruction: {instruction}\nResponse: {output_text}"
    return {"text": text}

formatted_data = raw_data["train"].map(format_example)
formatted_data[0]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

{'instruction': 'Summarize or explain the following regulatory/legal text in the context of cybersecurity compliance.',
 'input': 'Article 10',
 'output': '### Article 10 - Computer security incident response teams (CSIRTs)\nEach Member State shall designate or establish one or more CSIRTs. The CSIRTs may be designated or established within a competent authority. The CSIRTs shall comply with the requirements set out in\nArticle 11(1)\n, shall cover at least the sectors, subsectors and types of entity referred to in Annexes I and II, and shall be responsible for incident handling in accordance with a well-defined process.\nMember States shall ensure that each CSIRT has adequate resources to carry out effectively its tasks as set out in\nArticle 11(3)\n.\nMember States shall ensure that each CSIRT has at its disposal an appropriate, secure, and resilient communication and information infrastructure through which to exchange information with essential and important entities and other rele

In [6]:
def preprocess(sample):
    tokenized = tokenizer(
        sample["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_data = formatted_data.map(
    preprocess,
    remove_columns=formatted_data.column_names
)

tokenized_data[0]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

{'input_ids': [16664,
  25,
  8116,
  5612,
  551,
  476,
  10339,
  279,
  2701,
  22515,
  47036,
  1467,
  304,
  279,
  2266,
  315,
  61442,
  8733,
  624,
  2505,
  25,
  13355,
  220,
  16,
  15,
  198,
  2582,
  25,
  16600,
  13355,
  220,
  16,
  15,
  481,
  17407,
  4763,
  10455,
  2033,
  7263,
  320,
  6412,
  30521,
  82,
  340,
  4854,
  12039,
  3234,
  4880,
  74124,
  476,
  5695,
  825,
  476,
  803,
  10006,
  30521,
  82,
  13,
  576,
  10006,
  30521,
  82,
  1231,
  387,
  23195,
  476,
  9555,
  2878,
  264,
  39783,
  11198,
  13,
  576,
  10006,
  30521,
  82,
  4880,
  25017,
  448,
  279,
  8502,
  738,
  700,
  304,
  198,
  16651,
  220,
  16,
  16,
  7,
  16,
  340,
  11,
  4880,
  3421,
  518,
  3245,
  279,
  25512,
  11,
  1186,
  9687,
  1087,
  323,
  4494,
  315,
  5387,
  13862,
  311,
  304,
  88620,
  288,
  358,
  323,
  7946,
  11,
  323,
  4880,
  387,
  8480,
  369,
  10455,
  11589,
  304,
  18353,
  448,
  264,
  1632],
 'attention_mask':

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

c:\Users\miron\anaconda3\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [11]:
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator,
)

torch.cuda.empty_cache()
trainer.train()

Step,Training Loss


TrainOutput(global_step=6, training_loss=2.6371450424194336, metrics={'train_runtime': 32.6273, 'train_samples_per_second': 1.287, 'train_steps_per_second': 0.184, 'total_flos': 41746000942080.0, 'train_loss': 2.6371450424194336, 'epoch': 3.0})

In [12]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

('./my_qwen_lora\\tokenizer_config.json',
 './my_qwen_lora\\chat_template.jinja',
 './my_qwen_lora\\tokenizer.json')

In [13]:
from peft import PeftModel, PeftConfig

path = output_dir

config = PeftConfig.from_pretrained(path)

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

inference_model = PeftModel.from_pretrained(base_model, path)
inference_tokenizer = AutoTokenizer.from_pretrained(path, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [16]:
prompt = "Summarize Article 21 of NIS2"
inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)

with torch.no_grad():
    output = inference_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=120,
        do_sample=False,
    )

print(inference_tokenizer.decode(output[0], skip_special_tokens=True))

Summarize Article 21 of NIS2023

Article 21 of the National Information Security Standard (NIS) 2023 is a crucial section that outlines the responsibilities and obligations of organizations in protecting personal data. It emphasizes the importance of implementing strong security measures to prevent unauthorized access, data breaches, and other cyber threats. The article also highlights the need for regular updates and maintenance of these security measures to ensure their effectiveness against evolving cyber risks.

The article stresses the significance of conducting thorough risk assessments and vulnerability analyses to identify potential weaknesses in an organization's systems and processes. This assessment should be conducted regularly to stay informed
